# Bacterial analysis

## Import data.

In [3]:
!qiime tools import \
        --type "SampleData[PairedEndSequencesWithQuality]" \
        --input-path manifest.txt \
        --output-path raw_input.qza \
        --input-format PairedEndFastqManifestPhred33V2

Imported manifest.txt as PairedEndFastqManifestPhred33V2 to raw_input.qza


In [4]:
!qiime demux summarize \
        --i-data raw_input.qza \
        --p-n 10000 \
        --verbose \
        --o-visualization raw_input.qzv

/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:182: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  context['result_data'] = pd.concat([context['result_data'], df])
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['250 nts' '250 nts' '250 nts' '250 nts' '250 nts' '250 nts' '250 nts']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  stats[stats.index != 'count'] = \
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_s

## Remove primers, only keep reads with primers

In [5]:
!qiime cutadapt trim-paired \
  --i-demultiplexed-sequences raw_input.qza \
  --p-cores 10 \
  --p-front-f NTGYCAGCMGCCGCGGTAA \
  --p-front-r NCGYCAATTYMTTTRAGTTT \
  --p-discard-untrimmed \
  --o-trimmed-sequences trimmed_seqs.qza \
  --verbose > trim.log

Done           00:00:00        66,103 reads @   7.2 µs/read;   8.33 M reads/minute
Done           00:00:00        62,417 reads @   7.3 µs/read;   8.19 M reads/minute
Done           00:00:00        66,543 reads @   6.9 µs/read;   8.68 M reads/minute
Done           00:00:00        65,009 reads @   7.1 µs/read;   8.42 M reads/minute
Done           00:00:00        62,928 reads @   7.3 µs/read;   8.23 M reads/minute
Done           00:00:00        63,616 reads @   7.3 µs/read;   8.23 M reads/minute
Done           00:00:00        63,092 reads @   7.2 µs/read;   8.38 M reads/minute
Done           00:00:00        65,248 reads @   7.0 µs/read;   8.55 M reads/minute
Done           00:00:00        65,953 reads @   7.1 µs/read;   8.50 M reads/minute
Done           00:00:00        66,549 reads @   6.9 µs/read;   8.69 M reads/minute
Done           00:00:00        64,584 reads @   7.5 µs/read;   7.98 M reads/minute
Done           00:00:00        62,957 reads @   7.7 µs/read;   7.75 M reads/minute
Done

In [6]:
!qiime demux summarize \
     --i-data trimmed_seqs.qza \
     --p-n 10000 \
     --verbose \
     --o-visualization trimmed_seqs.qzv

/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:182: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  context['result_data'] = pd.concat([context['result_data'], df])
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['219 nts' '220 nts' '221 nts' '221 nts' '222 nts' '223 nts' '223 nts']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  stats[stats.index != 'count'] = \
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_s

## Denoise/deduplicate merge seqs

In [7]:
!qiime dada2 denoise-paired \
    --i-demultiplexed-seqs trimmed_seqs.qza \
    --p-trunc-len-f  0 \
    --p-trunc-len-r 0 \
    --p-n-threads 10 \
    --o-table 2023_dada_table.qza \
    --o-representative-sequences 2023_dada_sequences.qza \
    --o-denoising-stats 2023_dada_stats.qza \
    --o-base-transition-stats 2023_dada_transitions.qza \
    --verbose >> dada.log



Loading required package: Rcpp
/Users/ac044747/anaconda3/envs/qiime2-2026.4/lib/python3.12/site-packages/rachis/metadata/metadata.py:610: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 na

In [8]:
!qiime metadata tabulate \
    --m-input-file 2023_dada_stats.qza \
    --o-visualization 2023_dada_stats.qzv

!qiime feature-table summarize \
    --i-table 2023_dada_table.qza \
    --output-dir dadaRes \
    --m-metadata-file bacteriaMetadata.txt

Saved Visualization to: 2023_dada_stats.qzv
Saved ImmutableMetadata to: dadaRes/feature_frequencies.qza
Saved ImmutableMetadata to: dadaRes/sample_frequencies.qza
Saved Visualization to: dadaRes/summary.qzv


## Filter

Remove the Makiki samples that had very low DNA and were repeated.

In [10]:
!qiime feature-table filter-samples \
  --i-table 2023_dada_table.qza \
  --m-metadata-file bacteriaMetadata.txt \
  --p-where "Remove=0" \
  --o-filtered-table 2023_dada_table_filter_samples_repeat.qza

!qiime feature-table summarize \
    --i-table 2023_dada_table_filter_samples_repeat.qza \
    --output-dir dadaRes2 \
    --m-metadata-file bacteriaMetadata.txt

Saved FeatureTable[Frequency] to: 2023_dada_table_filter_samples_repeat.qza
Saved ImmutableMetadata to: dadaRes2/feature_frequencies.qza
Saved ImmutableMetadata to: dadaRes2/sample_frequencies.qza
Saved Visualization to: dadaRes2/summary.qzv


Remove samples from plants that were neither Preferred nor Avoided.

In [12]:
!qiime feature-table filter-samples \
  --i-table 2023_dada_table_filter_samples_repeat.qza \
  --m-metadata-file bacteriaMetadata.txt \
  --p-where "Preference!='UndefinedP'" \
  --o-filtered-table 2023_dada_table_filter_samples_UndefinedP.qza

!qiime feature-table summarize \
    --i-table 2023_dada_table_filter_samples_UndefinedP.qza \
    --output-dir dadaRes3 \
    --m-metadata-file bacteriaMetadata.txt

Saved FeatureTable[Frequency] to: 2023_dada_table_filter_samples_UndefinedP.qza
Saved ImmutableMetadata to: dadaRes3/feature_frequencies.qza
Saved ImmutableMetadata to: dadaRes3/sample_frequencies.qza
Saved Visualization to: dadaRes3/summary.qzv


Remove ASVs that are present in a single sample or that have a total frequency lower than 0.1% of the average read depth of the samples (~30, average reads per sample = 27,107)

In [13]:
!qiime feature-table filter-features \
  --i-table 2023_dada_table_filter_samples_UndefinedP.qza \
  --p-min-samples 2 \
  --p-min-frequency 30 \
  --o-filtered-table 2023_filtered_table.qza
!qiime feature-table summarize \
    --i-table 2023_filtered_table.qza \
    --output-dir dadaFiltered \
    --m-metadata-file bacteriaMetadata.txt

Saved FeatureTable[Frequency] to: 2023_filtered_table.qza
Saved ImmutableMetadata to: dadaFiltered/feature_frequencies.qza
Saved ImmutableMetadata to: dadaFiltered/sample_frequencies.qza
Saved Visualization to: dadaFiltered/summary.qzv


In [14]:
!qiime feature-table filter-seqs \
  --i-data 2023_dada_sequences.qza \
  --i-table 2023_filtered_table.qza \
  --o-filtered-data 2023_filtered_sequences.qza


Saved FeatureData[Sequence] to: 2023_filtered_sequences.qza
